In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("ecommerce_dataset_+1m.csv")

df['order_date'] = pd.to_datetime(df['order_date'])
df['account_creation_date'] = pd.to_datetime(df['account_creation_date'])

print(df['order_date'].dtype, df['account_creation_date'].dtype)

datetime64[us] datetime64[us]


In [2]:
df['is_returned'] = (df['order_status'] == 'Returned').astype(int)
df['is_cancelled'] = (df['order_status'] == 'Cancelled').astype(int)

# The "finalized" subset — the only orders that should count toward revenue/profit
df['is_finalized'] = df['order_status'].isin(['Completed', 'Returned']).astype(int)

print(df[['order_status', 'is_returned', 'is_cancelled', 'is_finalized']].drop_duplicates())

   order_status  is_returned  is_cancelled  is_finalized
0     Completed            0             0             1
5      Returned            1             0             1
12      Pending            0             0             0
38    Cancelled            0             1             0
58   Processing            0             0             0


In [3]:
# delivery_status is only meaningful for finalized orders (see explanation above)
df['delivery_status_reliable'] = df['is_finalized']

# Sanity check: how many "impossible" combos exist before we flag them?
impossible = df[(df['order_status'] == 'Cancelled') & (df['delivery_status'] == 'Delivered')]
print("Cancelled-but-Delivered rows:", len(impossible))

Cancelled-but-Delivered rows: 37663


In [4]:
# Recompute what total_price_usd SHOULD be, and compare to what's given
df['calc_gross'] = df['unit_price_usd'] * df['quantity']
df['calc_discount'] = df['calc_gross'] * (df['discount_percent'] / 100)
df['calc_total'] = df['calc_gross'] - df['calc_discount']

df['total_price_diff'] = (df['calc_total'] - df['total_price_usd']).abs()
print("Rows where calculated total differs from given total by more than $0.05:")
print((df['total_price_diff'] > 0.05).sum())

# Same check for profit
df['calc_profit'] = df['total_price_usd'] - df['cost_usd']
df['profit_diff'] = (df['calc_profit'] - df['profit_usd']).abs()
print("Rows where calculated profit differs from given profit by more than $0.05:")
print((df['profit_diff'] > 0.05).sum())

Rows where calculated total differs from given total by more than $0.05:
0
Rows where calculated profit differs from given profit by more than $0.05:
0


In [5]:
df['return_reason'] = df['return_reason'].fillna('Not Returned')
df['coupon_code'] = df['coupon_code'].fillna('No Coupon')
df['customer_feedback'] = df['customer_feedback'].fillna('No Feedback Given')

print(df[['return_reason', 'coupon_code', 'customer_feedback']].isnull().sum())  # should all be 0 now

return_reason        0
coupon_code          0
customer_feedback    0
dtype: int64


In [12]:
df.drop(columns=['calc_gross', 'calc_discount', 'calc_total', 'total_price_diff',
                  'calc_profit', 'profit_diff'], inplace=True, errors='ignore')

df.to_csv('Large_Clean', index=False)
print("Saved. Final shape:", df.shape)

Saved. Final shape: (1000123, 66)
